In [256]:
import matplotlib.pyplot as plt
from IPython import display
import numpy as np
import matplotlib.animation as animation
from qiskit.visualization import plot_state_qsphere, plot_histogram
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.quantum_info import DensityMatrix, Operator
from qiskit.circuit.library import Diagonal
from matplotlib.patches import Arc
import os

# initialize


In [271]:
# Parametri
target_state = '10101'
n = len(target_state)
N = 2**n

# Costruzione del circuito Grover
grover_circuit = QuantumCircuit(n)
all_qubits = grover_circuit.qubits
grover_circuit.h(all_qubits)

# Stato iniziale in sovrapposizione uniforme
state = Statevector.from_label('+' * len(all_qubits))

In [272]:

# Elenco delle directory da creare
dirs = [
    f'imgs/{target_state}',
    f'imgs/{target_state}/histogram',
    f'imgs/{target_state}/plot',
    f'imgs/{target_state}/qsphere',
    f'imgs/{target_state}/subspace',
    f'frames/{target_state}'
]

# Creazione delle directory se non esistono già
for dir_path in dirs:
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
        print(f"Cartella '{dir_path}' creata.")

    


## components

In [259]:
# Stato di marcatura
mark_state = Statevector.from_label(target_state)

hadamards = QuantumCircuit(n)
hadamards.h(all_qubits)
hadamards.name = 'Hadamards'  # Nome descrittivo del circuito


mark = QuantumCircuit(n)
mark.append(Diagonal((-1)**mark_state.data), range(n))
mark.name = 'Mark'  # Nome descrittivo del circuito

diffuse = QuantumCircuit(n)
diffuse.append(Diagonal((2 * DensityMatrix.from_label(n * '0') - Operator.from_label(n * 'I')).data.diagonal()), range(n))
diffuse.name = 'Diffuse'  # Nome descrittivo del circuito

## images

In [260]:
# Lista degli stati base
basis_states = [format(i, f'0{n}b') for i in range(2**n)]
    
# Trova l'indice corrispondente allo stato target |target_state⟩
index_target = basis_states.index(target_state)
    
    # Crea il vettore |target_state⟩
ket_target = np.zeros(2**n)
ket_target[index_target] = 1  # |target_state⟩ come vettore standard
    
    # Definisci il secondo asse come la somma normalizzata degli altri stati
ket_others = np.ones(2**n) - ket_target
ket_others = ket_others / np.linalg.norm(ket_others)  # Normalizzazione
    

# Array per salvare le immagini del sottospazio
subspace_images = []



def plot_state_in_subspace(state, step, operation):
    global ket_others, ket_target, target_state
    
    # Proiezioni dello stato quantistico sui due assi
    projection_target = np.dot(ket_target.conj(), state.data)  # Asse Y (target)
    projection_others = np.dot(ket_others.conj(), state.data)  # Asse X (others)
    
    projection_target = np.real(projection_target)
    projection_others = np.real(projection_others)

    # Calcolo dell'angolo theta in radianti
    theta = np.arctan2(projection_target, projection_others)  # theta = arctan(Y/X)

    # Plotting del vettore nel sottospazio 2D
    fig, ax = plt.subplots(figsize=(6, 6))

    # Plottiamo le componenti lungo gli assi X (others) e Y (target)
    ax.quiver(0, 0, projection_others, projection_target, angles='xy', scale_units='xy', scale=1, color='g', label=f'Vettore')

    # Configurazione del grafico
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.axhline(0, color='black', linewidth=0.5)  # Asse X
    ax.axvline(0, color='black', linewidth=0.5)  # Asse Y
    ax.grid(True)

    # Etichetta per l'angolo theta
    ax.text(0.5, 0.8, f'θ = {theta:.3f} rad', color='red', fontsize=12, ha='center', transform=ax.transAxes)
    
    radius = 0.4
    # Aggiunta dell'arco al grafico
    if 0 <= theta <= np.pi:
        # Se l'angolo è tra 0 e pi, l'arco parte dall'alto (angolo positivo)
        arco = Arc((0, 0), radius, radius, angle=0, theta1=0, theta2=np.degrees(theta), color='r', lw=2)
    else:
        # Se l'angolo è tra pi e 2pi, disegna l'arco dal basso
        arco = Arc((0, 0), radius, radius, angle=0, theta1=np.degrees(theta), theta2=360, color='r', lw=2)
    
    ax.add_patch(arco)

    # Legenda e titolo
    ax.legend()
    ax.set_title(f"Proiezione e angolo θ dopo {operation}, passo {step}")

    # Salva l'immagine
    filename_subspace = f'imgs/{target_state}/subspace/{step}_{operation}.png'
    plt.savefig(filename_subspace)
    plt.close(fig)

    # Aggiungi l'immagine all'array per la GIF
    subspace_images.append(filename_subspace)


In [261]:
q_sphere_images = []  # Per salvare le immagini della Q-sphere

def plot_state_as_sphere(state, step, operation):
    global target_state    
    # Q-sphere senza assi ai bordi
    fig, ax = plt.subplots(figsize=(5, 5))
    plot_state_qsphere(state, ax=ax)
    ax.axis('off')  # Rimuove gli assi
    ax.set_title(f"Dopo {operation}, passo {step}")
    filename_qsphere = f'imgs/{target_state}/qsphere/{step}_{operation}.png'
    plt.savefig(filename_qsphere)
    plt.close(fig)
    q_sphere_images.append(filename_qsphere)

In [262]:
sine_images = []

def plot_sine_with_vertical_line(state, step, operation):
    global ket_target, ket_others,target_state  # Supponiamo che ket_others rappresenti l'asse errata

    # Calcolo del prodotto scalare tra il vettore dello stato e l'asse errata (ket_others)
    dot_product = np.dot(ket_others.conj(), state.data)
    cos_theta = np.real(dot_product)  # Otteniamo il coseno dell'angolo
    
    # Calcoliamo l'angolo theta in radianti
    theta = np.arccos(cos_theta)
    
    # Definisci l'intervallo per la funzione seno
    x = np.linspace(0, 2 * np.pi, 10000)
    
    
    # Crea il grafico
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(x, np.sin(x)**2, label='sin(x)')
    ax.plot(x, np.cos(x)**2, label='cos(x)')
    
    # Plotta la linea verticale in corrispondenza di theta
    ax.axvline(x=theta, color='r', linestyle='--', label=f'θ = {theta:.2f} rad')

    # Aggiungi etichette e titolo
    ax.set_xlabel('x (radians)')
    ax.set_ylabel('sin(x)')
    ax.set_title(f"Seno con barra verticale dopo {operation}, passo {step}")
    ax.legend()
    
    # Indica θ sul grafico
    ax.text(theta, 0, f'θ = {theta:.2f}', color='red', ha='right', va='bottom', fontsize=12)
    
    # Salva l'immagine
    filename_sine = f'imgs/{target_state}/plot/{step}_{operation}.png'
    plt.savefig(filename_sine)
    plt.close(fig)
    
    # Aggiungi l'immagine all'array per la GIF
    sine_images.append(filename_sine)


In [263]:
histogram_images = []  # Per salvare le immagini dell'istogramma

def plot_state_as_histogram(state, step, operation):  
    global target_state  
    # Istogramma senza assi ai bordi e con asse y fisso su probabilità 1.00
    counts = state.probabilities_dict()
    fig, ax = plt.subplots(figsize=(6, 4))
    plot_histogram(counts, ax=ax)
    ax.set_ylim([0, 1.0])  # Imposta l'asse y da 0 a 1.0
    ax.set_title(f"Dopo {operation}, passo {step}")
    filename_histogram = f'imgs/{target_state}/histogram/{step}_{operation}.png'
    plt.savefig(filename_histogram)
    plt.close(fig)
    histogram_images.append(filename_histogram)

## gif

In [264]:

# Funzione per salvare gli stati e le probabilità
def save_frame(state, step, operation):    
    plot_state_in_subspace(state, step, operation)
    plot_state_as_sphere(state, step, operation)
    plot_state_as_histogram(state, step, operation)
    plot_sine_with_vertical_line(state, step, operation)


# Funzione per creare una GIF con una griglia 2x2
def create_gif(image_lists, gif_name, frames_dir='frames/'):
    global target_state
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    
    ims = []
    num_frames = max(len(image_list) for image_list in image_lists)  # Numero massimo di frame tra le liste
    
    # Loop sui frame
    for i in range(num_frames):
        # Pulizia delle immagini precedenti nei subplot
        for ax in axes.flatten():
            ax.clear()

        im_frame = []

        # Posizioni della griglia (2x2)
        positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

        # Loop sulle immagini da inserire nella griglia
        for j, image_list in enumerate(image_lists):
            if i < len(image_list):
                img_path = image_list[i]
                img = plt.imread(img_path)
                pos = positions[j]
                im_frame.append(axes[pos].imshow(img, animated=True))
                axes[pos].axis('off')
                #os.remove(img_path)  # Rimuove il file temporaneo dopo l'uso

        ims.append(im_frame)
        
        # Salva il frame come immagine PNG
        frame_filename = os.path.join(frames_dir, f'{target_state}/frame_{i:03d}.png')
        plt.savefig(frame_filename)

    # Creazione della GIF
    #ani = animation.ArtistAnimation(fig, ims, interval=500, repeat_delay=1000, blit=True)
    #ani.save(gif_name, writer='pillow')
    plt.close(fig)



# circuit

In [295]:
def print_probability(state,row,i = -1):
    if (i >= 0):
        print(f"Dopo {i+1} applicazioni di G:\n\tP(|{bin(row)[2:]}⟩) = {np.real(state.data[row])**2:>.4f}")
    else:
        print(f"Dopo inizializzazione:\n\tP(|{bin(row)[2:]}⟩) = {np.real(state.data[row])**2:>.4f}")
    
def print_best_row(state):
    global k
    best_row = np.argmax(state.data)
    print(f"\nTupla |{bin(best_row)[2:]}⟩ ottenuta con P = {np.real(state.data[best_row])**2:>.4f}:\n\t{{'Nome':'Mario','Cognome':'Rossi'}}")
    print(f"Oracolo utilizzato {k} volte")

In [296]:
#La tupla che viene marcata dall' oracolo
marked_state = 21 #|10101>

#Utilizzo la formula per ottenere il numero di iterazioni
k = round((np.pi / 4) * np.sqrt(N) - 1/2)

#Inizializzo i qubit tutti in |0⟩
state = Statevector.from_label('00000')

#rendo tutti gli stati equiprobabili
state = state.evolve(hadamards)

#Mostro la probabilità iniziale
print_probability(state,marked_state)

#Applico l' operatore di Grover k volte
for i in range(k):
    #Eseguo la marcatura tramite oracolo
    state = state.evolve(mark)    
    #Amplio le probabilitá SOLO sugli stati marcati
    state = state.evolve(hadamards).evolve(diffuse).evolve(hadamards)
    
    #Mostro la probabilità iniziale
    print_probability(state,marked_state,i)

#Mmostro la tupla con maggiore probabilitá
print_best_row(state)

    


Dopo inizializzazione:
	P(|10101⟩) = 0.0312
Dopo 1 applicazioni di G:
	P(|10101⟩) = 0.2583
Dopo 2 applicazioni di G:
	P(|10101⟩) = 0.6024
Dopo 3 applicazioni di G:
	P(|10101⟩) = 0.8969
Dopo 4 applicazioni di G:
	P(|10101⟩) = 0.9992

Tupla |10101⟩ ottenuta con P = 0.9992:
	{'Nome':'Mario','Cognome':'Rossi'}
Oracolo utilizzato 4 volte


In [267]:
import os
from PIL import Image

def create_gif_from_folders(folder_path, output_gif_name, duration=500):
    # Verifica che la cartella esista
    if not os.path.isdir(folder_path):
        raise ValueError(f"La cartella {folder_path} non esiste.")

    # Elenco delle sottocartelle
    subfolders = [os.path.join(folder_path, subfolder) for subfolder in sorted(os.listdir(folder_path)) if os.path.isdir(os.path.join(folder_path, subfolder))]
    
    # Verifica che ci siano esattamente 4 sottocartelle
    if len(subfolders) != 4:
        raise ValueError("Ci devono essere esattamente 4 sottocartelle nella cartella principale.")

    # Lista di immagini per ciascuna sottocartella
    images_per_folder = []

    for subfolder in subfolders:
        # Ordina e carica tutte le immagini nella sottocartella
        file_list = sorted(os.listdir(subfolder))
        images = []
        for file_name in file_list:
            file_path = os.path.join(subfolder, file_name)
            if os.path.isfile(file_path) and file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif')):
                img = Image.open(file_path)
                images.append(img)
        if not images:
            raise ValueError(f"Nessuna immagine trovata nella sottocartella {subfolder}.")
        images_per_folder.append(images)

    # Determina il numero massimo di frame in base al numero di immagini nelle sottocartelle
    max_frames = max(len(images) for images in images_per_folder)

    # Crea i frame combinando le immagini da ciascuna sottocartella
    gif_frames = []
    for i in range(max_frames):
        # Per ogni frame, prendi l'immagine corrispondente da ciascuna sottocartella
        images_to_combine = []
        for images in images_per_folder:
            # Se una cartella ha meno immagini, usa l'ultima immagine disponibile
            img = images[i] if i < len(images) else images[-1]
            images_to_combine.append(img)
        
        # Unisci le immagini in un frame 2x2 (forma quadrata)
        widths, heights = zip(*(img.size for img in images_to_combine))
        max_width = max(widths)
        max_height = max(heights)

        # Creiamo un'immagine che possa contenere una griglia 2x2
        total_width = max_width * 2  # 2 immagini di larghezza
        total_height = max_height * 2  # 2 immagini di altezza
        new_frame = Image.new('RGB', (total_width, total_height))

        # Incolla le 4 immagini in una griglia 2x2
        new_frame.paste(images_to_combine[0], (0, 0))                    # Posizione in alto a sinistra
        new_frame.paste(images_to_combine[1], (max_width, 0))            # Posizione in alto a destra
        new_frame.paste(images_to_combine[2], (0, max_height))           # Posizione in basso a sinistra
        new_frame.paste(images_to_combine[3], (max_width, max_height))   # Posizione in basso a destra

        gif_frames.append(new_frame)

    # Salva le immagini combinate come GIF
    gif_frames[0].save(output_gif_name, save_all=True, append_images=gif_frames[1:], duration=duration, loop=0)
    
    print(f"GIF salvata come {output_gif_name}")

# Esempio di utilizzo
folder_path = f'imgs/{target_state}'  # Modifica con il percorso della tua cartella di immagini
output_gif_name = f'gifs/{target_state}.gif'
create_gif_from_folders(folder_path, output_gif_name, duration=500)


GIF salvata come gifs/10101.gif


In [268]:
# Converti il circuito in un operatore
operator = Operator(grover_circuit)

# Recupera la matrice dell'operatore
matrix = operator.data

tolerance = 1e-10
matrix_clean = np.real(np.where(np.abs(matrix) < tolerance, 0, matrix))

#print("Matrice pulita del circuito:")
print(matrix_clean)

# Verifica dell'unitarietà
unitary_check = np.allclose(np.dot(matrix_clean.conj().T, matrix_clean), np.eye(matrix_clean.shape[0]))



[[ 0.1767767  0.1767767  0.1767767 ...  0.1767767  0.1767767  0.1767767]
 [ 0.1767767 -0.1767767  0.1767767 ... -0.1767767  0.1767767 -0.1767767]
 [ 0.1767767  0.1767767 -0.1767767 ...  0.1767767 -0.1767767 -0.1767767]
 ...
 [ 0.1767767 -0.1767767  0.1767767 ...  0.1767767 -0.1767767  0.1767767]
 [ 0.1767767  0.1767767 -0.1767767 ... -0.1767767  0.1767767  0.1767767]
 [ 0.1767767 -0.1767767 -0.1767767 ...  0.1767767  0.1767767 -0.1767767]]
